# Permutation Test — Random Forest

**What this answers:**  
Is RF's CV AUC of 0.8389 genuinely above chance, or could a model get that score by luck on this dataset even with random labels?

**How it works:**  
1. Compute the observed CV AUC with real labels (0.8389 — already known)
2. Shuffle `y_train` randomly 500 times, recompute CV AUC each time
3. The 500 shuffled AUCs form the **null distribution** — what AUC looks like when there is no real signal
4. p-value = fraction of null AUCs ≥ observed AUC (the `+1` correction avoids p=0)

**Pre-commitment (important):**  
500 shuffles was decided before seeing the results. This prevents post-hoc adjustment of the threshold to make the result look better.

**Why n_estimators=100 for permuted runs:**  
Each permuted run only needs to sample the null distribution accurately — it doesn't need the most precise AUC. 100 trees per fold is enough. The observed AUC was computed with 500 trees (that's what we're testing against).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pathlib, pickle

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

ROOT      = pathlib.Path().resolve().parent
SPLIT_DIR = ROOT / 'data' / 'processed' / 'split'

X_train = np.load(SPLIT_DIR / 'X_train.npy')
y_train = np.load(SPLIT_DIR / 'y_train.npy')

print(f'X_train : {X_train.shape}')
print(f'y_train : {y_train.shape}  (healthy={(y_train==0).sum()}  cancer={(y_train==1).sum()})')

## Run permutation test — 500 shuffles

In [ ]:
N_PERMUTATIONS = 500
OBSERVED_AUC   = 0.8389   # tuned RF CV AUC from notebook 02

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Faster RF for null distribution (100 trees is enough to sample the null)
rf_perm = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    max_features='sqrt',
    min_samples_leaf=4,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

rng       = np.random.default_rng(0)
null_aucs = np.zeros(N_PERMUTATIONS)

for i in range(N_PERMUTATIONS):
    y_perm      = rng.permutation(y_train)
    null_aucs[i] = cross_val_score(rf_perm, X_train, y_perm,
                                   cv=cv, scoring='roc_auc', n_jobs=-1).mean()
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{N_PERMUTATIONS}  null mean so far: {null_aucs[:i+1].mean():.4f}')

# p-value with +1 correction (Ojala & Garriga 2010)
p_value = (np.sum(null_aucs >= OBSERVED_AUC) + 1) / (N_PERMUTATIONS + 1)

print()
print(f'Observed AUC : {OBSERVED_AUC}')
print(f'Null mean    : {null_aucs.mean():.4f}')
print(f'Null max     : {null_aucs.max():.4f}')
print(f'p-value      : {p_value:.4f}  ({np.sum(null_aucs >= OBSERVED_AUC)}/{N_PERMUTATIONS} null AUCs ≥ observed)')

## Plot null distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(null_aucs, bins=40, color='#90A4AE', edgecolor='white', label='Null distribution (500 permutations)')
ax.axvline(OBSERVED_AUC, color='#E53935', linewidth=2.5,
           label=f'Observed AUC = {OBSERVED_AUC}  (p = {p_value:.4f})')
ax.axvline(0.5, color='#424242', linewidth=1, linestyle='--', label='Random chance (AUC=0.5)')

ax.set_xlabel('CV AUC (5-fold)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Permutation Test — Random Forest', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(ROOT / 'results' / 'permutation_test_rf.png', dpi=150)
plt.show()

print(f'Saved: results/permutation_test_rf.png')

---
# Permutation Test — CNN

**Why single split instead of 5-fold CV:**  
5-fold CV × 500 permutations × 30 epochs = ~6 hours on CPU. Instead we use a single stratified 80/20 hold-out split of the training data for each permuted run. The null distribution shape is identical — it still centers at 0.5 and bounds the maximum achievable AUC under random labels. We only need to sample it, not estimate each null AUC precisely.

**Pre-commitment (500 shuffles, same as RF).**

**30 epochs per permuted run** — the fast-converging folds (1, 3, 4) all stopped at 10–14 epochs. 30 is a safe upper bound that keeps each run to ~2 seconds on CPU.

**Observed CNN AUC = 0.8990** (5-fold CV from notebook 04).

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import pandas as pd

DEVICE         = torch.device('cpu')
CNN_EPOCHS     = 30
CNN_BATCH      = 32
CNN_LR         = 1e-3
CNN_OBSERVED   = 0.8990
N_PERM_CNN     = 500


class CfDNA_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 8,  kernel_size=15, padding=7),
            nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(8, 16, kernel_size=7,  padding=3),
            nn.ReLU(), nn.MaxPool1d(4),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5), nn.Linear(304, 32), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.fc(self.conv(x)).squeeze(1)


# Load and scale histogram features
df        = pd.read_csv(ROOT / 'data' / 'processed' / 'features_model.csv')
hist_cols = sorted([c for c in df.columns if c.startswith('hist_')])
X_hist    = df[hist_cols].astype(float).values
y_all     = (df['label'] == 'cancer').astype(int).values

# Same train split as notebook 04
X_tr_full, _, y_tr_full, _ = train_test_split(
    X_hist, y_all, test_size=0.20, stratify=y_all, random_state=42
)
scaler_h = StandardScaler().fit(X_tr_full)
X_tr_sc  = scaler_h.transform(X_tr_full)

# Fixed inner split for permutation runs: 80/20 of training data
X_in, X_val, y_in, y_val = train_test_split(
    X_tr_sc, y_tr_full, test_size=0.20, stratify=y_tr_full, random_state=0
)

def train_and_eval_cnn(X_tr, y_tr, X_v, y_v, seed):
    torch.manual_seed(seed)
    m   = CfDNA_CNN().to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=CNN_LR)
    pw  = torch.tensor([(y_tr == 0).sum() / (y_tr == 1).sum()], dtype=torch.float32)
    lf  = nn.BCEWithLogitsLoss(pos_weight=pw)
    Xt  = torch.tensor(X_tr[:, None, :], dtype=torch.float32)
    yt  = torch.tensor(y_tr, dtype=torch.float32)
    for _ in range(CNN_EPOCHS):
        m.train()
        for xb, yb in DataLoader(TensorDataset(Xt, yt), batch_size=CNN_BATCH, shuffle=True):
            opt.zero_grad(); lf(m(xb), yb).backward(); opt.step()
    m.eval()
    with torch.no_grad():
        proba = torch.sigmoid(m(torch.tensor(X_v[:, None, :], dtype=torch.float32))).numpy()
    return roc_auc_score(y_v, proba)

rng_cnn      = np.random.default_rng(1)
null_aucs_cnn = np.zeros(N_PERM_CNN)

for i in range(N_PERM_CNN):
    y_perm           = rng_cnn.permutation(y_in)
    null_aucs_cnn[i] = train_and_eval_cnn(X_in, y_perm, X_val, y_val, seed=i)
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{N_PERM_CNN}  null mean so far: {null_aucs_cnn[:i+1].mean():.4f}')

p_value_cnn = (np.sum(null_aucs_cnn >= CNN_OBSERVED) + 1) / (N_PERM_CNN + 1)

print()
print(f'Observed CNN AUC : {CNN_OBSERVED}')
print(f'Null mean        : {null_aucs_cnn.mean():.4f}')
print(f'Null max         : {null_aucs_cnn.max():.4f}')
print(f'p-value          : {p_value_cnn:.4f}  ({np.sum(null_aucs_cnn >= CNN_OBSERVED)}/{N_PERM_CNN} null AUCs ≥ observed)')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(null_aucs_cnn, bins=40, color='#90A4AE', edgecolor='white',
        label='Null distribution (500 permutations)')
ax.axvline(CNN_OBSERVED, color='#E53935', linewidth=2.5,
           label=f'Observed AUC = {CNN_OBSERVED}  (p = {p_value_cnn:.4f})')
ax.axvline(0.5, color='#424242', linewidth=1, linestyle='--', label='Random chance (AUC=0.5)')

ax.set_xlabel('AUC (single hold-out split)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Permutation Test — 1D CNN', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(ROOT / 'results' / 'permutation_test_cnn.png', dpi=150)
plt.show()

print('Saved: results/permutation_test_cnn.png')